In [12]:
# =============================================================================
# 01_setup_and_acquisition
# CALSHIFT: calibration source and distribution shift in conformal NIDS
# Cell 1 - bootstrap: mount Drive, restore git credentials
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, shutil, hashlib, subprocess, zipfile, io, urllib.request
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PARENT_DIR   = DRIVE_ROOT / 'CALSHIFT_Research'
PROJECT_ROOT = PARENT_DIR / 'calshift-research'
DATASETS_DIR = DRIVE_ROOT / 'NIDS_Datasets'
CRED_DIR     = DRIVE_ROOT / '.gitcreds'

for fname, dest in [('.git-credentials', '/root/.git-credentials'),
                    ('.gitconfig',       '/root/.gitconfig')]:
    src = CRED_DIR / fname
    if src.exists():
        shutil.copy(src, dest)
        os.chmod(dest, 0o600)
        print('restored', fname)
    else:
        print('MISSING', src, '- create it once, then re-run')

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
PARENT_DIR.mkdir(parents=True, exist_ok=True)
print('drive ready')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MISSING /content/drive/MyDrive/.gitcreds/.git-credentials - create it once, then re-run
MISSING /content/drive/MyDrive/.gitcreds/.gitconfig - create it once, then re-run
drive ready


In [13]:
# =============================================================================
# Cell 2 - clone or update the repository
# Create the EMPTY repo on GitHub first: anasbiswas1/calshift-research
# Keep it private. Do not add collaborators yet.
# =============================================================================
GITHUB_USER = 'anasbiswas1'
REPO_NAME   = 'calshift-research'
REPO_URL    = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'

if not (PROJECT_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL], cwd=str(PARENT_DIR), check=True)

os.chdir(PROJECT_ROOT)
subprocess.run(['git', 'config', 'user.name',  'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', 'user.email', 'anasbiswas@gmail.com'], check=True)
subprocess.run(['git', 'pull', '--ff-only'], check=False)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('repo root:', os.getcwd())

repo root: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [14]:
# =============================================================================
# Cell 3 - canonical folder layout and gitignore
# =============================================================================
for d in ['notebooks', 'src', 'reports', 'figures', 'data', 'data/interim', 'data/processed']:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

GITIGNORE = """
data/
*.pkl
*.joblib
*.parquet
*.npz
.ipynb_checkpoints/
__pycache__/
*.pyc
.DS_Store
"""
(PROJECT_ROOT / '.gitignore').write_text(GITIGNORE.lstrip())

for d in ['reports', 'figures']:
    (PROJECT_ROOT / d / '.gitkeep').touch()

print(sorted(p.name for p in PROJECT_ROOT.iterdir() if not p.name.startswith('.')))

['data', 'figures', 'notebooks', 'reports', 'src']


In [15]:
# =============================================================================
# Cell 4 - write src/config.py
# Every constant here is fixed by preregistration.md. Nothing is tuned later.
# =============================================================================
CONFIG_LINES = [
    '"""Project configuration. Values fixed by preregistration.md; do not tune."""',
    'from pathlib import Path',
    'import math',
    '',
    'DRIVE_ROOT   = Path("/content/drive/MyDrive")',
    'PARENT_DIR   = DRIVE_ROOT / "CALSHIFT_Research"',
    'PROJECT_ROOT = PARENT_DIR / "calshift-research"',
    'DATASETS_DIR = DRIVE_ROOT / "NIDS_Datasets"',
    '',
    'DATA_DIR    = PROJECT_ROOT / "data"',
    'INTERIM_DIR = DATA_DIR / "interim"',
    'PROC_DIR    = DATA_DIR / "processed"',
    'REPORTS_DIR = PROJECT_ROOT / "reports"',
    'FIGURES_DIR = PROJECT_ROOT / "figures"',
    '',
    '# preregistration section 5',
    'SEEDS = [42, 1337, 2024, 7, 91, 512, 6021, 88, 3407, 12345]',
    '',
    '# preregistration section 7.5',
    'ALPHA_PRIMARY     = 0.05',
    'ALPHA_SENSITIVITY = [0.10, 0.20]',
    'ALPHA_CONDITIONAL = [0.01]',
    '',
    '# preregistration section 4, stratified split of the SOURCE partition',
    'SPLIT_FRACTIONS = {"train": 0.60, "val": 0.10, "probcal": 0.15, "source_cal_pool": 0.15}',
    '',
    '# preregistration section 7.6',
    'def min_calib_n(alpha):',
    '    return math.ceil(1.0 / alpha) - 1',
    '',
    '# preregistration section 9',
    'SCOV_SUBSAMPLE_PER_SIDE = 20000',
    'PERMUTATION_NULL_DRAWS  = 200',
    'PERMUTATION_NULL_Q      = 0.95',
    '',
    '# preregistration sections 8.1 and 10.1',
    'N_MATCHED_DRAWS       = 10',
    'N_LADDER_REALIZATIONS = {"nslkdd": 20, "cicids2017": 5, "ugr16": 5, "ciciot2023": 5}',
    '',
    '# preregistration section 10.2',
    'MIN_ESS_FOCAL_CLASS = 30',
    '',
    '# preregistration section 13.1',
    'PRACTICAL_COVERAGE_DROP = 0.05',
    '',
    'TRAIN_ROW_CAP = 2000000',
    '',
    'CANONICAL_CLASSES = ["Normal", "DoS", "Probe", "R2L", "U2R"]',
]
(PROJECT_ROOT / 'src' / 'config.py').write_text(chr(10).join(CONFIG_LINES) + chr(10))
(PROJECT_ROOT / 'src' / '__init__.py').touch()

import importlib
if 'config' in sys.modules:
    importlib.reload(sys.modules['config'])
import config
print('seeds:', config.SEEDS)
print('min calib n at alpha 0.05:', config.min_calib_n(0.05))
print('min calib n at alpha 0.01:', config.min_calib_n(0.01))

seeds: [42, 1337, 2024, 7, 91, 512, 6021, 88, 3407, 12345]
min calib n at alpha 0.05: 19
min calib n at alpha 0.01: 99


In [16]:
# =============================================================================
# Cell 5 - acquisition helpers
# Hashes are RECORDED on first run and VERIFIED on every later run.
# No hardcoded hashes: we do not trust a hash we cannot independently confirm.
# =============================================================================
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def fetch(url, dest: Path, timeout=120):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print('present:', dest.name)
        return dest
    print('downloading:', url)
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=timeout) as r, open(dest, 'wb') as out:
        shutil.copyfileobj(r, out)
    print('saved:', dest, f'{dest.stat().st_size/1e6:.1f} MB')
    return dest

def require_manual(dest: Path, instruction: str):
    if dest.exists():
        return True
    print('MANUAL STEP REQUIRED')
    print('  expected file :', dest)
    print('  instruction   :', instruction)
    return False

HASHES_PATH = config.REPORTS_DIR / 'dataset_hashes.json'
HASHES = json.loads(HASHES_PATH.read_text()) if HASHES_PATH.exists() else {}

def register_hash(key: str, path: Path):
    h = sha256_of(path)
    if key in HASHES:
        status = 'OK' if HASHES[key] == h else 'MISMATCH'
        print(f'{status:9s} {key}')
        if status == 'MISMATCH':
            raise RuntimeError(f'hash changed for {key}: expected {HASHES[key]}, got {h}')
    else:
        HASHES[key] = h
        print(f'{"RECORDED":9s} {key}')
    return h

print('helpers ready')

helpers ready


In [17]:
# =============================================================================
# Cell 6 - NSL-KDD
# Role: positive control. Deliberate benchmark mismatch, KDDTest+ contains
# attack subtypes absent from KDDTrain+. Subtypes are RETAINED because the
# ladder in notebook 03 manipulates unseen-subtype fraction.
# =============================================================================
NSL_DIR = config.DATASETS_DIR / 'nsl-kdd'
NSL_DIR.mkdir(parents=True, exist_ok=True)

NSL_SOURCES = {
    'KDDTrain+.txt': 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt',
    'KDDTest+.txt':  'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt',
}

for fname, url in NSL_SOURCES.items():
    dest = NSL_DIR / fname
    try:
        fetch(url, dest)
    except Exception as e:
        print('auto-download failed:', e)
        require_manual(dest, 'download from https://www.unb.ca/cic/datasets/nsl.html and place here')

for fname in NSL_SOURCES:
    p = NSL_DIR / fname
    if p.exists():
        register_hash(f'nslkdd/{fname}', p)

present: KDDTrain+.txt
present: KDDTest+.txt
RECORDED  nslkdd/KDDTrain+.txt
RECORDED  nslkdd/KDDTest+.txt


In [18]:
# =============================================================================
# Cell 7 - CIC-IDS2017, WTMC-2021 corrected (Engelen et al. 2021)
# The uncorrected UNB CSVs are NOT acceptable: flow-construction and labelling
# defects are documented, and using them would invite the exact criticism the
# audit levels at others.
# =============================================================================
CIC17_DIR = config.DATASETS_DIR / 'cicids2017-wtmc2021'
CIC17_DIR.mkdir(parents=True, exist_ok=True)

# Set this once you have the corrected release URL or a Drive copy.
# Source of record: https://intrusion-detection.distrinet-research.be/WTMC2021/
CIC17_URL = None

CIC17_ARCHIVE = CIC17_DIR / 'wtmc2021_corrected.zip'

if CIC17_URL:
    try:
        fetch(CIC17_URL, CIC17_ARCHIVE)
    except Exception as e:
        print('auto-download failed:', e)

csvs = sorted(CIC17_DIR.glob('**/*.csv'))
if not csvs and CIC17_ARCHIVE.exists():
    with zipfile.ZipFile(CIC17_ARCHIVE) as z:
        z.extractall(CIC17_DIR)
    csvs = sorted(CIC17_DIR.glob('**/*.csv'))

if not csvs:
    require_manual(
        CIC17_DIR / '<corrected day CSVs>',
        'get the WTMC-2021 corrected CSVs from '
        'https://intrusion-detection.distrinet-research.be/WTMC2021/ '
        'and place them under this folder'
    )
else:
    print(f'{len(csvs)} corrected CSV files found')
    for p in csvs:
        register_hash(f'cicids2017/{p.name}', p)

MANUAL STEP REQUIRED
  expected file : /content/drive/MyDrive/NIDS_Datasets/cicids2017-wtmc2021/<corrected day CSVs>
  instruction   : get the WTMC-2021 corrected CSVs from https://intrusion-detection.distrinet-research.be/WTMC2021/ and place them under this folder


In [19]:
# =============================================================================
# Cell 8 - NSL-KDD canonical schema
# Emits broad class AND subtype. Subtype is required by the ladder.
# =============================================================================
import pandas as pd
import numpy as np

NSL_COLUMNS = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
    'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate',
    'dst_host_srv_rerror_rate','subtype','difficulty'
]

NSL_SUBTYPE_TO_CLASS = {
    'normal': 'Normal',
    'back':'DoS','land':'DoS','neptune':'DoS','pod':'DoS','smurf':'DoS',
    'teardrop':'DoS','apache2':'DoS','udpstorm':'DoS','processtable':'DoS',
    'mailbomb':'DoS','worm':'DoS',
    'satan':'Probe','ipsweep':'Probe','nmap':'Probe','portsweep':'Probe',
    'mscan':'Probe','saint':'Probe',
    'guess_passwd':'R2L','ftp_write':'R2L','imap':'R2L','phf':'R2L',
    'multihop':'R2L','warezmaster':'R2L','warezclient':'R2L','spy':'R2L',
    'xlock':'R2L','xsnoop':'R2L','snmpguess':'R2L','snmpgetattack':'R2L',
    'httptunnel':'R2L','sendmail':'R2L','named':'R2L',
    'buffer_overflow':'U2R','loadmodule':'U2R','rootkit':'U2R','perl':'U2R',
    'sqlattack':'U2R','xterm':'U2R','ps':'U2R',
}

def load_nslkdd(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=NSL_COLUMNS)
    df['subtype'] = df['subtype'].str.strip().str.lower()
    unknown = sorted(set(df['subtype']) - set(NSL_SUBTYPE_TO_CLASS))
    if unknown:
        raise ValueError(f'unmapped NSL-KDD subtypes: {unknown}')
    df['label'] = df['subtype'].map(NSL_SUBTYPE_TO_CLASS)
    return df.drop(columns=['difficulty'])

nsl_train = load_nslkdd(NSL_DIR / 'KDDTrain+.txt')
nsl_test  = load_nslkdd(NSL_DIR / 'KDDTest+.txt')

config.INTERIM_DIR.mkdir(parents=True, exist_ok=True)
nsl_train.to_parquet(config.INTERIM_DIR / 'nslkdd_train.parquet', index=False)
nsl_test.to_parquet(config.INTERIM_DIR / 'nslkdd_test.parquet', index=False)

pd.DataFrame(sorted(NSL_SUBTYPE_TO_CLASS.items()), columns=['subtype','broad_class'])\
  .to_csv(config.REPORTS_DIR / 'nslkdd_subtype_mapping.csv', index=False)

print('train', nsl_train.shape, 'test', nsl_test.shape)
print(nsl_train['label'].value_counts().to_dict())
print(nsl_test['label'].value_counts().to_dict())

train (125973, 43) test (22544, 43)
{'Normal': 67343, 'DoS': 45927, 'Probe': 11656, 'R2L': 995, 'U2R': 52}
{'Normal': 9711, 'DoS': 7460, 'R2L': 2885, 'Probe': 2421, 'U2R': 67}


In [20]:
# =============================================================================
# Cell 9 - NSL-KDD subtype support analysis
# This is the raw material for the ladder. Records which subtypes appear only
# in test, with their counts, so notebook 03 can build randomized realizations
# at a fixed unseen-subtype fraction.
# =============================================================================
train_sub = set(nsl_train['subtype'])
test_sub  = set(nsl_test['subtype'])

unseen = sorted(test_sub - train_sub)
shared = sorted(test_sub & train_sub)

unseen_tbl = (nsl_test[nsl_test['subtype'].isin(unseen)]
              .groupby(['label','subtype']).size()
              .rename('n_test').reset_index()
              .sort_values(['label','n_test'], ascending=[True, False]))

print(f'subtypes in train: {len(train_sub)}')
print(f'subtypes in test : {len(test_sub)}')
print(f'test-only        : {len(unseen)}')
print()
print(unseen_tbl.to_string(index=False))

unseen_share = nsl_test['subtype'].isin(unseen).mean()
print(f'\ntest mass in unseen subtypes: {unseen_share:.4f}')

unseen_tbl.to_csv(config.REPORTS_DIR / 'nslkdd_unseen_subtypes.csv', index=False)

subtypes in train: 23
subtypes in test : 38
test-only        : 17

label       subtype  n_test
  DoS       apache2     737
  DoS  processtable     685
  DoS      mailbomb     293
  DoS      udpstorm       2
  DoS          worm       2
Probe         mscan     996
Probe         saint     319
  R2L     snmpguess     331
  R2L snmpgetattack     178
  R2L    httptunnel     133
  R2L         named      17
  R2L      sendmail      14
  R2L         xlock       9
  R2L        xsnoop       4
  U2R            ps      15
  U2R         xterm      13
  U2R     sqlattack       2

test mass in unseen subtypes: 0.1663


In [21]:
# =============================================================================
# Cell 10 - CIC-IDS2017 canonical schema
# Labels are DISCOVERED, not hardcoded. The corrected release introduces
# "Attempted" variants; both the raw label and a normalised family are kept so
# that the treatment decision is made explicitly and recorded, not implied.
# =============================================================================
def load_cicids2017(csv_paths):
    frames = []
    for p in csv_paths:
        d = pd.read_csv(p, low_memory=False)
        d.columns = [c.strip() for c in d.columns]
        lab_col = next(c for c in d.columns if c.lower() == 'label')
        d = d.rename(columns={lab_col: 'label_raw'})
        d['label_raw'] = d['label_raw'].astype(str).str.strip()
        d['source_file'] = p.name
        frames.append(d)
    return pd.concat(frames, ignore_index=True)

if csvs:
    cic17 = load_cicids2017(csvs)
    cic17['is_attempted'] = cic17['label_raw'].str.contains('Attempt', case=False, na=False)
    cic17['label_family'] = (cic17['label_raw']
                             .str.replace(r'\s*-\s*Attempted', '', regex=True)
                             .str.strip())

    lab_tbl = (cic17.groupby(['source_file','label_raw'])
               .size().rename('n').reset_index()
               .sort_values(['source_file','n'], ascending=[True, False]))
    lab_tbl.to_csv(config.REPORTS_DIR / 'cicids2017_label_inventory.csv', index=False)

    print('rows:', len(cic17))
    print('distinct raw labels:', cic17['label_raw'].nunique())
    print('attempted rows:', int(cic17['is_attempted'].sum()))
    print()
    print(cic17['label_family'].value_counts().head(20).to_string())

    cic17.to_parquet(config.INTERIM_DIR / 'cicids2017_wtmc2021.parquet', index=False)
else:
    print('SKIPPED: corrected CSVs not present. Complete Cell 7 first.')

SKIPPED: corrected CSVs not present. Complete Cell 7 first.


In [22]:
# =============================================================================
# Cell 11 - OPEN DECISION, must be resolved before notebook 03
# The corrected CIC-IDS2017 release separates completed from attempted attacks.
# preregistration.md does not state how "Attempted" rows are treated. That is a
# gap. Three defensible options; the choice must be written into the
# preregistration BEFORE any coverage number is computed.
#
#   A. exclude Attempted rows entirely
#   B. treat Attempted as its own class
#   C. merge Attempted into the parent attack family
#
# Recommendation: A for the primary analysis, C as sensitivity. Attempted
# attacks are behaviourally distinct and Engelen et al. separated them
# deliberately; folding them in would blur the class-conditional analysis that
# the paper's claim depends on.
# =============================================================================
ATTEMPTED_POLICY = None   # set to 'exclude' | 'separate' | 'merge'

if ATTEMPTED_POLICY is None:
    print('BLOCKED: set ATTEMPTED_POLICY, then add the decision to preregistration.md')
else:
    print('policy:', ATTEMPTED_POLICY)

BLOCKED: set ATTEMPTED_POLICY, then add the decision to preregistration.md


In [23]:
# =============================================================================
# Cell 12 - dataset manifest
# =============================================================================
manifest = {'datasets': {}}

if 'nsl_train' in globals():
    manifest['datasets']['nslkdd'] = {
        'role': 'positive control, deliberate benchmark mismatch',
        'train_rows': int(len(nsl_train)),
        'test_rows': int(len(nsl_test)),
        'train_class_counts': {k: int(v) for k, v in nsl_train['label'].value_counts().items()},
        'test_class_counts':  {k: int(v) for k, v in nsl_test['label'].value_counts().items()},
        'n_subtypes_train': int(len(train_sub)),
        'n_subtypes_test': int(len(test_sub)),
        'n_subtypes_test_only': int(len(unseen)),
        'test_mass_unseen_subtypes': float(unseen_share),
    }

if 'cic17' in globals():
    manifest['datasets']['cicids2017'] = {
        'role': 'session and day aware, corrected (WTMC-2021)',
        'rows': int(len(cic17)),
        'n_files': int(len(csvs)),
        'n_labels_raw': int(cic17['label_raw'].nunique()),
        'attempted_rows': int(cic17['is_attempted'].sum()),
        'attempted_policy': globals().get('ATTEMPTED_POLICY'),
    }

manifest['hashes'] = HASHES
manifest['config'] = {
    'seeds': config.SEEDS,
    'alpha_primary': config.ALPHA_PRIMARY,
    'split_fractions': config.SPLIT_FRACTIONS,
    'n_matched_draws': config.N_MATCHED_DRAWS,
    'min_ess_focal_class': config.MIN_ESS_FOCAL_CLASS,
}

(config.REPORTS_DIR / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2))
HASHES_PATH.write_text(json.dumps(HASHES, indent=2))
print(json.dumps(manifest['datasets'], indent=2))

{
  "nslkdd": {
    "role": "positive control, deliberate benchmark mismatch",
    "train_rows": 125973,
    "test_rows": 22544,
    "train_class_counts": {
      "Normal": 67343,
      "DoS": 45927,
      "Probe": 11656,
      "R2L": 995,
      "U2R": 52
    },
    "test_class_counts": {
      "Normal": 9711,
      "DoS": 7460,
      "R2L": 2885,
      "Probe": 2421,
      "U2R": 67
    },
    "n_subtypes_train": 23,
    "n_subtypes_test": 38,
    "n_subtypes_test_only": 17,
    "test_mass_unseen_subtypes": 0.16634137686302342
  }
}


In [24]:
# =============================================================================
# Cell 13 - PROJECTED feasibility and focal class
# The binding table is produced in notebook 02 after real partitioning. This
# projection uses the preregistered split fractions and exists to catch a
# design-breaking asymmetry early.
#
# CRITICAL: feasibility must be evaluated for BOTH calibration sources.
#   SHC calibrates on the SOURCE pool  -> source_cal_pool fraction of train
#   TSC calibrates on the TARGET half  -> 0.5 of the evaluation partition
# A class feasible under TSC but not under SHC cannot support the paired
# TSC-vs-SHC contrast at all, because SHC has no valid quantile for it.
# =============================================================================
if 'nsl_train' in globals():
    f_src = config.SPLIT_FRACTIONS['source_cal_pool']
    shc_n = (nsl_train['label'].value_counts() * f_src).round().astype(int)
    tsc_n = (nsl_test['label'].value_counts() * 0.50).round().astype(int)

    alphas = [config.ALPHA_PRIMARY] + config.ALPHA_SENSITIVITY + config.ALPHA_CONDITIONAL
    rows = []
    for a in alphas:
        need = config.min_calib_n(a)
        for cls in config.CANONICAL_CLASSES:
            s, t = int(shc_n.get(cls, 0)), int(tsc_n.get(cls, 0))
            rows.append({
                'dataset': 'nslkdd', 'alpha': a, 'class': cls,
                'min_calib_needed': need,
                'shc_calib_projected': s, 'tsc_calib_projected': t,
                'shc_feasible': s >= need, 'tsc_feasible': t >= need,
                'contrast_possible': (s >= need) and (t >= need),
                'asymmetric': (t >= need) and (s < need),
            })
    proj = pd.DataFrame(rows)
    proj.to_csv(config.REPORTS_DIR / 'feasibility_projected_nslkdd.csv', index=False)
    print(proj.to_string(index=False))

    asym = proj[proj['asymmetric']]
    if len(asym):
        print()
        print('ASYMMETRIC CLASSES (TSC feasible, SHC not) - paired contrast impossible:')
        print(asym[['alpha', 'class', 'shc_calib_projected',
                    'tsc_calib_projected', 'min_calib_needed']].to_string(index=False))

    attacks = [c for c in config.CANONICAL_CLASSES if c != 'Normal']
    ok = proj[(proj['alpha'] == config.ALPHA_PRIMARY) &
              (proj['contrast_possible']) &
              (proj['class'].isin(attacks))]
    if len(ok):
        focal = ok.sort_values('shc_calib_projected').iloc[0]['class']
        print(f'\nPROJECTED FOCAL CLASS (nslkdd, alpha={config.ALPHA_PRIMARY}): {focal}')
        print('Rule: rarest attack class supporting the paired contrast in the source pool.')
        print('Confirm in notebook 02 against real partitions, then record before any coverage.')
    else:
        print('\nNO FEASIBLE FOCAL CLASS at primary alpha. Design must be revisited.')

dataset  alpha  class  min_calib_needed  shc_calib_projected  tsc_calib_projected  shc_feasible  tsc_feasible  contrast_possible  asymmetric
 nslkdd   0.05 Normal                19                10101                 4856          True          True               True       False
 nslkdd   0.05    DoS                19                 6889                 3730          True          True               True       False
 nslkdd   0.05  Probe                19                 1748                 1210          True          True               True       False
 nslkdd   0.05    R2L                19                  149                 1442          True          True               True       False
 nslkdd   0.05    U2R                19                    8                   34         False          True              False        True
 nslkdd   0.10 Normal                 9                10101                 4856          True          True               True       False
 nslkdd   0.1

In [26]:
# =============================================================================
# Cell 14 - persist git credentials, commit, push
# Nothing uncommitted overnight.
# =============================================================================
CRED_DIR.mkdir(parents=True, exist_ok=True)
for src, dst in [('/root/.git-credentials', CRED_DIR / '.git-credentials'),
                 ('/root/.gitconfig',       CRED_DIR / '.gitconfig')]:
    if os.path.exists(src):
        shutil.copy(src, dst)

os.chdir(PROJECT_ROOT)
subprocess.run(['git', 'add', '-A'], check=True)
status = subprocess.run(['git', 'status', '--porcelain'], capture_output=True, text=True).stdout
if status.strip():
    subprocess.run(['git', 'commit', '-m',
                    'nb01: setup, NSL-KDD and corrected CIC-IDS2017 acquisition, manifest'],
                   check=True)
    subprocess.run(['git', 'push'], check=True)
    print('pushed')
else:
    print('nothing to commit')

print(subprocess.run(['git', 'log', '--oneline', '-5'],
                     capture_output=True, text=True).stdout)

nothing to commit
2458ca9 nb01: setup, NSL-KDD and corrected CIC-IDS2017 acquisition, manifest

